In [ ]:
import pandas as pd
import sqlite3

db_path = "SQLite-db/sessions.db"
conn = sqlite3.connect(db_path)

sessions_df = pd.read_sql("""
SELECT * FROM digital_20250904_activation_sessions
""", conn)
sessions_df["ts"] = pd.to_datetime(sessions_df["ts"])
sessions_df["channel_id"] = sessions_df["channel_id"].astype(int)

# Gaps left in construction:

## 2.2 Synthetic unit pool preparation 
- Prefer synthetic viewer that have not yet been activated on the current broadcast day. 

I might have this working. I'm not confident I understand this instruction.

## 2.3 Activation and deactivation thresholds 
- Activation and deactivation are evaluated independently within each demographic bucket.


## 3.1 Linking while active 
- While a synthetic viewer is active, matching digital-linear sessions are logically linked to it. 

What does this mean??? Do I filter out sessions linked to inactive viewers?


## 3.2 Deactivation and reactivation
- Only after partially filled units are satisfied may empty or previously deactivated units be reused. 

I still don't understand what role activation status plays in linking new sessions. The organization of the instructions makes this unclear. Do I need to track activation status while linking viewers?

What do I do with the accumulated linked sessions when a viewer deactivates? The expected output structure has no "is_active" capacity. 

- Records must preserve channel, broadcast day, time interval, demographic bucket, synthetic unit identifier, and 
relevant representational weight fields needed for KPI calculation.

Two possiblities: 

1. Active/inactive status plays no further role in the output deliverable, and only is meant to affect linking assignments.
2. Only active users are carried to output. 

In [ ]:
import numpy as np
# Build sorted list of synthetic viewers from viewer_weights
# What this covers: Sort eligible viewer by:
# 1. Prefer viewers who were not logged in viewer_sessions on this date.
# 2. Ascending representational weight within each demographic bucket

date = "2025-09-04"
table_date = date.replace("-","")

activation_order_df = pd.read_sql(f"""
SELECT * FROM
(SELECT 
demo, viewer_id, viewer_weight,
rank() OVER (
    PARTITION BY demo ORDER BY activated_today, viewer_weight
    ) AS activation_order
FROM
(SELECT demo, dvm.viewer_id, viewer_weight,
CASE WHEN avl.viewer_id IS NULL THEN 0 ELSE 1 END AS activated_today
from digital_viewer_mapping dvm
LEFT JOIN active_viewer_list_{table_date} avl
ON dvm.viewer_id = avl.viewer_id))
order by activation_order;
""", conn)
conn.commit()
conn.close()

activation_order_df.head(7)

,demo,viewer_id,viewer_weight,activation_order
0,f1229,15341_18,1.027501,1
1,f3049,24565_30,4.218503,1
2,fover50,23326_30,3.044015,1
3,m1229,25501_28,1.029116,1
4,m3049,6316_30,2.681863,1
5,mover50,27599_30,2.994282,1
6,f1229,28246_27,1.032612,2


## Original build ran over 21 minutes. AI had improvements to speed it up:

1. Viewer lookup dict - Store (viewer_id, index) when linking, so "end" events use direct index access instead of filtering
2. reset_index(drop=True) - Cleaner indexing
3. .index[0] instead of .iloc[0].name - Faster
4. Remove DataFrame filtering on end events - Was doing links_df[links_df["viewer_id"] == v_id] every time

Expected improvement: 5-10x faster depending on your dataset size.


## Second set of speed-up suggestions:
Key speedups:

1. itertuples() instead of iterrows() - 3-5x faster (named tuple access vs row object creation)
2. np.where() for filtering - Faster than pandas boolean indexing
3. .iloc[] with column position - Direct memory access vs .loc[] label lookup
4. Remove nested dict structure - session_viewer_links[demo][channel] → session_viewer_links[demo]

Expected improvement: 10-15x faster overall compared to original code.

## Third Speedup:

1. Cache column indices - Eliminate .get_loc() calls (was doing this millions of times)
2. Use NumPy arrays - Direct memory access instead of pandas .iloc[]
3. Simplified dict key - (demo, s_id) instead of (demo, s_id, channel) **This is actually not workable in the model**
4. Direct array indexing - arrays['session_count'][ind] vs .iloc[ind, col_loc]

Expected improvement: 20-30% faster than before.

**This actually hugely helped. Speed is now 3,000/sec**

In [146]:
# Second AI speed-up:
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

from config import digital_sessions_target_groups
import numpy as np

start = time.time()
zeros = np.zeros(len(activation_order_df))
links_activation_order_df = pd.concat(
    [
    activation_order_df,
    pd.Series(zeros, name="session_count"), 
    pd.Series(zeros, name="active"), 
    ], 
    axis=1)
links_activation_order_df["channel"] = None

session_viewer_links = {}
links = {}
output_sessions = []

for demo in digital_sessions_target_groups:
    links[demo] = links_activation_order_df[links_activation_order_df["demo"]==demo].copy().reset_index(drop=True)

logger.info(f"Setup time: {time.time() - start:.2f}s")

loop_start = time.time()
processed = 0

# Convert to itertuples for ~3-5x faster iteration than iterrows
for row in sessions_df.itertuples():
    processed += 1
    if processed % 10000 == 0:
        elapsed = time.time() - loop_start
        rate = processed / elapsed
        remaining = (len(sessions_df) - processed) / rate
        logger.info(f"Progress: {processed}/{len(sessions_df)} {processed/len(sessions_df) * 100.0}% ({rate:.0f}/sec) ~{remaining:.0f}s left")
    
    s_id = row.session_id
    ts = row.ts
    demo = row.target_group
    channel = row.channel_id
    links_df = links[demo]

    if row.session_event == "start":

        # Vectorized filtering - much faster than chained boolean conditions
        # First, search for active viewers to attach to
        mask = ((links_df["channel"] == channel) | (links_df["channel"].isnull())) & \
               (links_df["session_count"] <= links_df["viewer_weight"] - 1) & \
               (links_df["active"] == 1)
        possible_indices = np.where(mask.values)[0]
        
        if len(possible_indices) > 0:
            link_viewer_ind = possible_indices[0] # Is it faster here to convert to python int with .item()?
            v_id = links_df.iloc[link_viewer_ind]["viewer_id"]

            links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")] += 1
            links_df.iloc[link_viewer_ind, links_df.columns.get_loc("channel")] = channel

            # (session_id, channel) is PK. demo is included to speed up search? Or is the PK relying on all 3 values?
            session_viewer_links[(demo, s_id, channel)] = (v_id, link_viewer_ind)

        # No active viewers found, pick inactive one
        else: 
            mask = ((links_df["channel"] == channel) | (links_df["channel"].isnull())) & \
                            (links_df["session_count"] <= links_df["viewer_weight"] - 1) & \
                            (links_df["active"] == 0)
            possible_indices = np.where(mask.values)[0]

            if len(possible_indices) == 0:
                logger.warning(f"No viewer found for session {s_id} in {demo}")

            links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")] += 1
            links_df.iloc[link_viewer_ind, links_df.columns.get_loc("channel")] = channel

            # (session_id, channel) is PK. demo is included to speed up search? Or is the PK relying on all 3 values?
            session_viewer_links[(demo, s_id, channel)] = (v_id, link_viewer_ind)

            # Check if activation
            if links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")] >= \
            round(links_df.iloc[link_viewer_ind, links_df.columns.get_loc("viewer_weight")] * 0.5):
                # Activate in links_df
                links_df.iloc[link_viewer_ind, links_df.columns.get_loc("active")] = 1

                # Log start of active session in output
                output_sessions.append((ts, channel, v_id, demo, 1))
        

    else:  # Session Finish
        try:
            v_id, link_viewer_ind = session_viewer_links[(demo, s_id, channel)]
        except KeyError:
            logger.warning(f"Viewer not found for session_id: {s_id}")
            raise

        links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")] -= 1
        
        # Check if deactivation
        if links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")] < \
        round(links_df.iloc[link_viewer_ind, links_df.columns.get_loc("viewer_weight")] * 0.5):
            # Deactivate in links_df
            links_df.iloc[link_viewer_ind, links_df.columns.get_loc("active")] = 0
            # Log end of active session in output
            output_sessions.append((ts, channel, v_id, demo, 0))

        if links_df.iloc[link_viewer_ind, links_df.columns.get_loc("session_count")].item() < 0:
            logger.error(f"Sub-zero session_count at {(ts, v_id, s_id)}")

logger.info(f"Total loop time: {time.time() - loop_start:.2f}s")

# Convert session_viewer_links dict to DataFrame and write to file.
output_sessions_df = pd.DataFrame(
    output_sessions,
    columns=["ts","channel_id","viewer_id","target_group","activation_flag"]
)

output_sessions_df.to_csv(f"output_sessions_{table_date}")

2026-08-15 19:03:05,852 - Setup time: 0.04s


2026-08-15 19:03:26,775 - Progress: 10000/1493964 0.6693601719987898% (478/sec) ~3105s left
2026-08-15 19:03:48,034 - Progress: 20000/1493964 1.3387203439975797% (474/sec) ~3109s left
2026-08-15 19:04:04,031 - Progress: 30000/1493964 2.0080805159963697% (516/sec) ~2839s left


KeyboardInterrupt: 

Activation logic:
Once activated, accumulates more views.
Once deactivated, is more likely to lose sessions than gain them and will quickly fall to empty.

In [179]:
# Third AI speedup
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

from config import digital_sessions_target_groups
import numpy as np

start = time.time()
zeros = np.zeros(len(activation_order_df))
links_activation_order_df = pd.concat(
    [
    activation_order_df,
    pd.Series(zeros, name="session_count"), 
    pd.Series(zeros, name="active"), 
    ], 
    axis=1)
links_activation_order_df["channel"] = None

session_viewer_links = {}
links = {}
output_sessions = []
links_arrays = {}  # Store as numpy arrays for faster access
col_idx = {}  # Cache column indices

for demo in digital_sessions_target_groups:
    df = links_activation_order_df[links_activation_order_df["demo"]==demo].copy().reset_index(drop=True)
    links[demo] = df
    
    # Cache column positions
    col_idx[demo] = {
        'session_count': df.columns.get_loc("session_count"),
        'channel': df.columns.get_loc("channel"),
        'viewer_id': df.columns.get_loc("viewer_id"),
        'viewer_weight': df.columns.get_loc("viewer_weight"),
        'active': df.columns.get_loc("active")
    }
    
    # Convert to numpy arrays with writeable flag
    links_arrays[demo] = {
        'session_count': np.array(df["session_count"].values, copy=True),
        'channel': np.array(df["channel"].values, copy=True),
        'viewer_id': np.array(df["viewer_id"].values, copy=True),
        'viewer_weight': np.array(df["viewer_weight"].values, copy=True),
        'active': np.array(df["active"].values, copy=True)
    }

logger.info(f"Setup time: {time.time() - start:.2f}s")

loop_start = time.time()
processed = 0

for row in sessions_df.itertuples():
    processed += 1
    if processed % 100000 == 0:
        elapsed = time.time() - loop_start
        rate = processed / elapsed
        remaining = (len(sessions_df) - processed) / rate
        logger.info(f"Progress: {processed}/{len(sessions_df)} ({rate:.0f}/sec) ~{remaining:.0f}s left")
    
    s_id = row.session_id
    ts = row.ts
    demo = row.target_group
    channel = row.channel_id
    links_df = links[demo]
    arrays = links_arrays[demo]
    idx = col_idx[demo]

    if row.session_event == "start":
        # Vectorized filtering
        mask = ((arrays['channel'] == channel) | (arrays['channel'] == None)) & \
               (arrays['session_count'] <= arrays['viewer_weight'] - 1) & \
               (arrays['active'] == 1)
        possible_indices = np.where(mask)[0]
        
        if len(possible_indices) > 0:
            link_viewer_ind = int(possible_indices[0])
            v_id = arrays['viewer_id'][link_viewer_ind]

            # Update arrays directly
            arrays['session_count'][link_viewer_ind] += 1
            arrays['channel'][link_viewer_ind] = channel
            
            session_viewer_links[(demo, s_id, channel)] = (v_id, link_viewer_ind)
        else:
            # Try inactive viewers
            mask = ((arrays['channel'] == channel) | (arrays['channel'] == None)) & \
                   (arrays['session_count'] <= arrays['viewer_weight'] - 1) & \
                   (arrays['active'] == 0)
            possible_indices = np.where(mask)[0]

            if len(possible_indices) > 0:
                link_viewer_ind = int(possible_indices[0])
                v_id = arrays['viewer_id'][link_viewer_ind]

                arrays['session_count'][link_viewer_ind] += 1
                arrays['channel'][link_viewer_ind] = channel
                
                session_viewer_links[(demo, s_id, channel)] = (v_id, link_viewer_ind)
                
                # Check Activation
                if arrays['session_count'][link_viewer_ind] >= round(arrays['viewer_weight'][link_viewer_ind] * 0.5):
                    arrays['active'][link_viewer_ind] = 1
                    output_sessions.append((ts, channel, v_id, demo, 1))
            else:
                logger.warning(f"No viewer found for session {s_id} in {demo}")

    else:  # Session Finish
        try:
            v_id, link_viewer_ind = session_viewer_links[(demo, s_id, channel)]
            arrays['session_count'][link_viewer_ind] -= 1

            # Check Deactivation
            if (arrays['session_count'][link_viewer_ind] < round(arrays['viewer_weight'][link_viewer_ind] * 0.5)) and \
                (arrays['active'][link_viewer_ind] == 1):
                arrays['active'][link_viewer_ind] = 0
                output_sessions.append((ts, channel, v_id, demo, 0))
            
            if arrays['session_count'][link_viewer_ind] < 0:
                logger.error(f"Sub-zero session_count at {(ts, v_id)}")
        except KeyError:
            logger.warning(f"Viewer not found for session_id: {s_id}")

logger.info(f"Total loop time: {time.time() - loop_start:.2f}s")


# Write output sessions to file and db
output_sessions_df = pd.DataFrame(
    output_sessions,
    columns=["ts","channel_id","viewer_id","target_group","activation_flag"]
)
output_sessions_df.to_sql(name=f"output_sessions_{table_date}", if_exists="replace", con=conn)
output_sessions_df.to_csv(f"output_sessions_{table_date}")

2026-08-15 19:58:52,946 - Setup time: 0.04s


2026-08-15 19:59:24,448 - Progress: 100000/1493964 (3174/sec) ~439s left
2026-08-15 19:59:58,764 - Progress: 200000/1493964 (3039/sec) ~426s left
2026-08-15 20:00:34,689 - Progress: 300000/1493964 (2949/sec) ~405s left
2026-08-15 20:01:08,799 - Progress: 400000/1493964 (2944/sec) ~372s left
2026-08-15 20:01:45,457 - Progress: 500000/1493964 (2898/sec) ~343s left
2026-08-15 20:02:22,589 - Progress: 600000/1493964 (2862/sec) ~312s left
2026-08-15 20:02:56,412 - Progress: 700000/1493964 (2875/sec) ~276s left
2026-08-15 20:03:29,601 - Progress: 800000/1493964 (2892/sec) ~240s left
2026-08-15 20:04:05,383 - Progress: 900000/1493964 (2881/sec) ~206s left
2026-08-15 20:04:36,969 - Progress: 1000000/1493964 (2907/sec) ~170s left
2026-08-15 20:05:06,646 - Progress: 1100000/1493964 (2944/sec) ~134s left
2026-08-15 20:05:39,450 - Progress: 1200000/1493964 (2952/sec) ~100s left
2026-08-15 20:06:08,190 - Progress: 1300000/1493964 (2987/sec) ~65s left
2026-08-15 20:06:36,894 - Progress: 1400000/1493

CSV Output structure
channel, broadcast day, time interval, demographic bucket, synthetic unit identifier, representational weight 

channel - session_viewer_links_df
time_interval - session_viewer_links_df
demo - session_viewer_links_df
viewer_id - session_viewer_links_df
weight - 


In [ ]:
# Transform Active Session Log to Pivoted Viewer Session Table
from config import model_version
conn = sqlite3.connect(db_path)

try:
    conn.execute(f"DELETE FROM output_active_sessions WHERE tv_date = '{date}'")
except:
    logger.info("Table output_active_sessions doesn't exist yet, creating...")
    conn.execute(f"""
        CREATE TABLE output_active_sessions AS
        select 
            '{date}' AS tv_date,
            channel_id, sesh.target_group, sesh.viewer_id, session_start, session_finish,
            round((julianday(session_finish) - julianday(session_start))* 86400, 0) AS session_duration,
            vw.viewer_weight
        FROM
            (select 
            channel_id,
            viewer_id,
            target_group,
            MAX(CASE WHEN activation_flag = 1 THEN ts END) AS session_start,
            MAX(CASE WHEN activation_flag = 0 THEN ts END) AS session_finish
            from 
                (select 
                datetime(ts) AS ts, channel_id, viewer_id, target_group, activation_flag,
                sum(activation_flag) OVER  
                (PARTITION BY viewer_id order by ts 
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_index
                from output_sessions_{table_date})
            group by 
                channel_id,viewer_id,target_group,session_index) sesh
            LEFT JOIN  (SELECT DISTINCT viewer, viewer_weight FROM viewer_weights
                WHERE model_version = '{model_version}') vw
                ON vw.viewer = sesh.viewer_id
        """)

conn.execute(f"""
INSERT INTO output_active_sessions
select 
    '{date}' AS tv_date,
    channel_id, sesh.target_group, sesh.viewer_id, session_start, session_finish,
    round((julianday(session_finish) - julianday(session_start))* 86400, 0) AS session_duration,
    vw.viewer_weight
FROM
    (select 
    channel_id,
    viewer_id,
    target_group,
    MAX(CASE WHEN activation_flag = 1 THEN ts END) AS session_start,
    MAX(CASE WHEN activation_flag = 0 THEN ts END) AS session_finish
    from 
        (select 
        datetime(ts) AS ts, channel_id, viewer_id, target_group, activation_flag,
        sum(activation_flag) OVER  
        (PARTITION BY viewer_id order by ts 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_index
        from output_sessions_{table_date})
    group by 
        channel_id,viewer_id,target_group,session_index) sesh
    LEFT JOIN  (SELECT DISTINCT viewer, viewer_weight FROM viewer_weights
    WHERE model_version = '{model_version}') vw
    ON vw.viewer = sesh.viewer_id
""")

conn.commit()
conn.close()

2026-08-15 20:52:36,791 - Table output_active_sessions doesn't exist yet, creating...
